# 🍽️ Restaurant Rating Prediction
### Zomato Bangalore Dataset — Full ML Project

## Step 1: Import All Libraries

In [ ]:
!pip install matplotlib seaborn scikit-learn gradio joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, classification_report, confusion_matrix)
import joblib
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

## Step 2: Load the Dataset

In [ ]:
data = pd.read_csv("zomato.csv")

print("Dataset loaded successfully!")
print("Shape of dataset (rows, columns):", data.shape)

## Step 3: First Look at the Data

In [ ]:
# See the first 5 rows
data.head()

In [ ]:
# See all column names
print("Column names:")
print(data.columns.tolist())

In [ ]:
# Check data types and how many values are missing
data.info()

## Step 4: Data Cleaning

In [ ]:
# The 'rate' column looks like "4.1/5" — we only need the number before '/'
# Also remove rows where rating is 'NEW' or '-' since they have no value

data = data[data['rate'] != 'NEW']
data = data[data['rate'] != '-']

# Extract number before '/' and convert to float
data['rate'] = data['rate'].astype(str).apply(lambda x: x.split('/')[0])
data['rate'] = data['rate'].astype(float)

print("Cleaned 'rate' column. Sample values:")
print(data['rate'].head())

In [ ]:
# The cost column has commas like "1,200" — remove them so we can use it as a number
data['approx_cost(for two people)'] = (
    data['approx_cost(for two people)']
    .astype(str)
    .str.replace(',', '')
    .str.strip()
)
data['approx_cost(for two people)'] = pd.to_numeric(
    data['approx_cost(for two people)'], errors='coerce'
)

print("Cleaned cost column. Sample values:")
print(data['approx_cost(for two people)'].head())

In [ ]:
# Check how many missing values are in each column
print("Missing values per column:")
print(data.isnull().sum())

## Step 5: Select Features and Target Variable

In [ ]:
# Features = inputs we give the model
# Target   = what we want to predict (the rating)

features = data[['online_order', 'book_table', 'votes',
                  'approx_cost(for two people)', 'listed_in(type)']]
target = data['rate']

# Combine features and target, then remove rows with missing values
dataset = pd.concat([features, target], axis=1).dropna()

print("Final dataset shape after dropping missing rows:", dataset.shape)

# Separate X (inputs) and y (output)
X = dataset.drop('rate', axis=1)
y = dataset['rate']

## Step 6: Exploratory Data Analysis (EDA)
EDA helps us understand the data before building the model.

In [ ]:
# Distribution of restaurant ratings
plt.figure(figsize=(8, 4))
sns.histplot(y, bins=20, kde=True, color='steelblue')
plt.title('Distribution of Restaurant Ratings')
plt.xlabel('Rating')
plt.ylabel('Number of Restaurants')
plt.tight_layout()
plt.show()

In [ ]:
# Does accepting online orders affect the rating?
plt.figure(figsize=(7, 4))
sns.boxplot(x='online_order', y='rate', data=dataset, palette='Set2')
plt.title('Online Order vs Restaurant Rating')
plt.xlabel('Accepts Online Orders')
plt.ylabel('Rating')
plt.tight_layout()
plt.show()

In [ ]:
# Does table booking affect the rating?
plt.figure(figsize=(7, 4))
sns.boxplot(x='book_table', y='rate', data=dataset, palette='Set3')
plt.title('Table Booking vs Restaurant Rating')
plt.xlabel('Allows Table Booking')
plt.ylabel('Rating')
plt.tight_layout()
plt.show()

In [ ]:
# Average rating by restaurant type
plt.figure(figsize=(10, 5))
type_rating = dataset.groupby('listed_in(type)')['rate'].mean().sort_values(ascending=False)
sns.barplot(x=type_rating.index, y=type_rating.values, palette='viridis')
plt.title('Average Rating by Restaurant Type')
plt.xlabel('Restaurant Type')
plt.ylabel('Average Rating')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Do restaurants with more votes get higher ratings?
plt.figure(figsize=(8, 4))
plt.scatter(dataset['votes'], y, alpha=0.3, color='coral')
plt.title('Votes vs Rating')
plt.xlabel('Number of Votes')
plt.ylabel('Rating')
plt.tight_layout()
plt.show()

## Step 7: Encode Categorical Variables
ML models only understand numbers, so we convert text columns into numbers.

In [ ]:
# pd.get_dummies converts Yes/No and type columns into 0s and 1s
X_encoded = pd.get_dummies(X, drop_first=True)
X_encoded = X_encoded.astype(int)

print("Encoded feature shape:", X_encoded.shape)
X_encoded.head()

## Step 8: Split Data into Training and Testing Sets

In [ ]:
# 80% of data is used to train the model
# 20% is kept aside to test how well the model performs

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")

## Step 9: Train Multiple Models and Compare
We train 4 different models to see which performs best.

In [ ]:
# Model 1: Linear Regression (simple baseline)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

lr_mae  = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2   = r2_score(y_test, lr_pred)

print(f"Linear Regression → MAE: {lr_mae:.3f} | RMSE: {lr_rmse:.3f} | R²: {lr_r2:.3f}")

In [ ]:
# Model 2: Decision Tree
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

dt_mae  = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
dt_r2   = r2_score(y_test, dt_pred)

print(f"Decision Tree     → MAE: {dt_mae:.3f} | RMSE: {dt_rmse:.3f} | R²: {dt_r2:.3f}")

In [ ]:
# Model 3: Random Forest (ensemble of many decision trees)
rf_model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2   = r2_score(y_test, rf_pred)

print(f"Random Forest     → MAE: {rf_mae:.3f} | RMSE: {rf_rmse:.3f} | R²: {rf_r2:.3f}")

In [ ]:
# Model 4: Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)

gb_mae  = mean_absolute_error(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_r2   = r2_score(y_test, gb_pred)

print(f"Gradient Boosting → MAE: {gb_mae:.3f} | RMSE: {gb_rmse:.3f} | R²: {gb_r2:.3f}")

## Step 10: Model Comparison Table and Chart

In [ ]:
# Show all model results in one table
comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting'],
    'MAE':   [lr_mae,  dt_mae,  rf_mae,  gb_mae],
    'RMSE':  [lr_rmse, dt_rmse, rf_rmse, gb_rmse],
    'R²':    [lr_r2,   dt_r2,   rf_r2,   gb_r2]
})

print("Model Comparison Table:")
print(comparison_df.to_string(index=False))

In [ ]:
# Bar chart comparing R² scores — higher is better
plt.figure(figsize=(8, 4))
sns.barplot(x='Model', y='R²', data=comparison_df, palette='Blues_d')
plt.title('Model Comparison — R² Score (Higher is Better)')
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Step 11: Feature Importance Plot
Shows which input features have the most impact on the predicted rating.

In [ ]:
# Get importance scores from the Random Forest model
importances = rf_model.feature_importances_
feature_names = X_encoded.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).head(10)  # show top 10

plt.figure(figsize=(9, 5))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='rocket')
plt.title('Top 10 Most Important Features (Random Forest)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

## Step 12: Classification — Good vs Bad Restaurant
We classify restaurants as Good (rating ≥ 3.5) or Bad (rating < 3.5).

In [ ]:
# Convert ratings to binary: 1 = Good, 0 = Bad
y_class = (y >= 3.5).astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_encoded, y_class, test_size=0.2, random_state=42
)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_c, y_train_c)
y_pred_class = log_model.predict(X_test_c)

print("Classification Report:")
print(classification_report(y_test_c, y_pred_class,
                             target_names=['Bad (<3.5)', 'Good (>=3.5)']))

In [ ]:
# Confusion matrix shows correct and wrong predictions
cm = confusion_matrix(y_test_c, y_pred_class)
cm_df = pd.DataFrame(cm,
                     index=['Actual Bad', 'Actual Good'],
                     columns=['Predicted Bad', 'Predicted Good'])

print("Confusion Matrix:")
print(cm_df)

# Plot as heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

## Step 13: Save the Model

In [ ]:
# Save the trained model so we can use it later without retraining
joblib.dump(rf_model, "restaurant_rating_model.pkl")
joblib.dump(X_encoded.columns.tolist(), "model_features.pkl")

print("Model saved as 'restaurant_rating_model.pkl'")
print("Feature list saved as 'model_features.pkl'")

## Step 14: Gradio Frontend (User Interface)
A simple web interface where users can enter restaurant details and get a predicted rating.

In [ ]:
import gradio as gr
import joblib
import pandas as pd
import numpy as np

# Load the saved model and feature list
model = joblib.load("restaurant_rating_model.pkl")
feature_columns = joblib.load("model_features.pkl")

def predict_rating(votes, cost, online_order, book_table, rest_type):
    # Start with a blank row of zeros
    input_df = pd.DataFrame(
        np.zeros((1, len(feature_columns))),
        columns=feature_columns
    )

    # Fill in values from the user
    if 'votes' in input_df.columns:
        input_df['votes'] = votes

    for col in input_df.columns:
        if 'cost' in col.lower():
            input_df[col] = cost

    if online_order == "Yes" and 'online_order_Yes' in input_df.columns:
        input_df['online_order_Yes'] = 1

    if book_table == "Yes" and 'book_table_Yes' in input_df.columns:
        input_df['book_table_Yes'] = 1

    type_col = f'listed_in(type)_{rest_type}'
    if type_col in input_df.columns:
        input_df[type_col] = 1

    # Make prediction
    prediction = model.predict(input_df)[0]
    rating = round(float(prediction), 2)

    # Give a label based on predicted rating
    if rating >= 4.0:
        label = "Excellent Restaurant!"
    elif rating >= 3.5:
        label = "Good Restaurant"
    elif rating >= 3.0:
        label = "Average Restaurant"
    else:
        label = "Below Average"

    return f"Predicted Rating: {rating} / 5.0 — {label}"

# Build the Gradio interface
interface = gr.Interface(
    fn=predict_rating,
    inputs=[
        gr.Number(label="Number of Votes", value=100),
        gr.Number(label="Average Cost for Two (Rs)", value=500),
        gr.Radio(["Yes", "No"], label="Online Order Available?", value="Yes"),
        gr.Radio(["Yes", "No"], label="Table Booking Available?", value="No"),
        gr.Dropdown(
            ["Delivery", "Dine-out", "Desserts", "Cafes", "Drinks & nightlife", "Buffet"],
            label="Restaurant Type",
            value="Delivery"
        )
    ],
    outputs=gr.Textbox(label="Prediction Result"),
    title="Restaurant Rating Predictor",
    description="Enter restaurant details to predict its Zomato rating using Machine Learning."
)

interface.launch()